# 🎞️ Anima on Diffucore UI — Colab (free T4)

Two cells, in order. Nothing to install by hand, nothing to run out of sequence:

1. **1️⃣ Set up** — Drive + all the settings (token, password, defaults). ~10 seconds.
2. **2️⃣ 🚀 Start everything** — installs what's missing, downloads the model once,
   boots the UI, opens the public link, loads Anima with the T4 speed stack and warms
   the GPU up. Finished steps are skipped, so it is always safe to re-run.

Then: **3️⃣** adds LoRAs / checkpoints anytime · **4️⃣** stops the server or shows its log.

*(In a hurry? Colab's **Runtime → Run all** works too — every cell is safe in order.)*

You need a free [ngrok token](https://dashboard.ngrok.com/get-started/your-authtoken): paste it in cell 1,
or add it as a Colab Secret named `NGROK_TOKEN`.

Models and images live in `MyDrive/AnimaDiffucore/`, so they survive across sessions.
On a free T4 expect **20–40 s per image** at 832×1216 (~8–15 s with the turbo LoRA).

> ⚠️ The ngrok link is public — set `UI_PASSWORD` in cell 1 to gate it.


In [ ]:
# @title 1️⃣ Set up — Drive, link, defaults {"form-width":"30%"}
USE_DRIVE = True                  # @param {type:"boolean"}
DRIVE_FOLDER = "AnimaDiffucore"   # @param {type:"string"}
NGROK_TOKEN = ""                  # @param {type:"string"}  free token: dashboard.ngrok.com/get-started/your-authtoken
UI_PASSWORD = ""                  # @param {type:"string"}  set one to protect the public link
NGROK_DOMAIN = ""                 # @param {type:"string"}  optional static ngrok domain
DEFAULT_NEG = "score_1, score_2, score_3, bad anatomy, bad proportions, deformed anatomy, deformed face, deformed eyes, deformed fingers, fat , loli, young, censored , too skinny"  # @param {type:"string"}  pre-filled negative prompt (clear it to leave the UI's own value)

FETCH_MODELS = True               # @param {type:"boolean"}  fetch the base model files (files already in Drive are skipped)
TURBO_LORA = True                 # @param {type:"boolean"}  also fetch the 8–12 step turbo LoRA
FACE_DETAILER = True              # @param {type:"boolean"}  also fetch the face-fix model
INSTALL_DEPS = True               # @param {type:"boolean"}  install/update Diffucore + Python deps

PRELOAD = True                    # @param {type:"boolean"}  load Anima for you (skip to load from the UI bar)
FAST_PARAMS = True                # @param {type:"boolean"}  20-step defaults + CFG interval in the form
WARMUP = True                     # @param {type:"boolean"}  one throwaway image → your first click is instant
KEEP_LORAS_APPLIED = True         # @param {type:"boolean"}  speed: keep an unchanged <lora:…> set fused between images (skips a multi-second re-stage)
FAST_TURING_KERNELS = True        # @param {type:"boolean"}  int8 attention + fused glue (T4)
RESIDENT_ENCODERS = True          # @param {type:"boolean"}  keep TE+VAE on the GPU (False only if you hit OOM)
USE_TORCH_COMPILE = False         # @param {type:"boolean"}  slow first run and drops int8 — leave off

# ------------------------- nothing to edit below -----------------------------
import os, re, shutil, subprocess, time
from pathlib import Path
import requests
from tqdm.auto import tqdm

if USE_DRIVE:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        print("Mounting Google Drive …")
        drive.mount("/content/drive")
    GDRIVE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
else:
    GDRIVE = Path("/content/anima_local")   # ephemeral: wiped when the runtime resets

DRIVE_MODELS, DRIVE_OUTPUTS = GDRIVE / "models", GDRIVE / "outputs"
for _sub in ("checkpoints", "diffusion-models", "vae", "text-encoders",
             "loras", "detailers", "upscalers", "oss_cache", "teacache_cache"):
    (DRIVE_MODELS / _sub).mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)

UI_DIR = Path("/content/diffucore-ui")   # engine + UI code (re-installed each session)
PORT = 7860

HF_ANIMA = "https://huggingface.co/circlestone-labs/Anima/resolve/main"
HF_ANIMA_LORAS = "https://huggingface.co/circlestone-labs/Anima-Official-LoRAs/resolve/main"
HF_ADETAILER = "https://huggingface.co/Bingsu/adetailer/resolve/main"


def _remote_info(url: str):
    """(content-disposition, content-length) via HEAD; ('', 0) on failure."""
    try:
        r = requests.head(url, allow_redirects=True, timeout=30)
        return (r.headers.get("Content-Disposition", ""),
                int(r.headers.get("Content-Length") or 0))
    except Exception:
        return "", 0


def _resolve_filename(url: str, content_disposition: str) -> str:
    m = re.search(r'filename="?([^";]+)"?', content_disposition)
    if m:
        return m.group(1)
    # Some CDNs answer HEAD without Content-Disposition: peek at a GET's headers.
    try:
        r = requests.get(url, stream=True, allow_redirects=True, timeout=30)
        cd = r.headers.get("Content-Disposition", "")
        r.close()
        m = re.search(r'filename="?([^";]+)"?', cd)
        if m:
            return m.group(1)
    except Exception:
        pass
    return url.split("?")[0].rstrip("/").split("/")[-1] or "download.bin"


def _aria2c(url: str, part: Path) -> bool:
    """Parallel download with aria2 (-c resumes the .part file). True on success."""
    if shutil.which("aria2c") is None:
        return False
    cmd = ["aria2c", "-c", "-x16", "-s16", "-k", "1M", "--file-allocation=none",
           "--console-log-level=warn", "--summary-interval=10",
           "--retry-wait=3", "--max-tries=10",
           "--timeout=60", "--connect-timeout=30",
           "-d", str(part.parent), "-o", part.name, url]
    try:
        rc = subprocess.run(cmd).returncode
    except Exception:
        return False
    return rc == 0 and part.exists() and part.stat().st_size > 0


def _stream_to(r, part, resume, total, filename):
    with open(part, "ab" if resume else "wb") as f, tqdm(
            total=total or None, initial=resume, unit="B",
            unit_scale=True, desc=filename) as bar:
        for chunk in r.iter_content(1 << 20):
            f.write(chunk)
            bar.update(len(chunk))


def fetch(url: str, dest_dir, filename=None) -> Path:
    """Downloader: aria2 16-connection parallel when available, else streaming
    requests. Both resume partial .part files and skip files already on disk."""
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    cd, expected = _remote_info(url)
    if filename is None:
        filename = _resolve_filename(url, cd)
    final = dest_dir / filename
    part = dest_dir / (filename + ".part")
    if final.exists() and (expected == 0 or final.stat().st_size == expected):
        print(f"\u2713 already present: {filename}")
        return final
    if _aria2c(url, part):
        part.rename(final)
        print(f"\u2713 downloaded: {filename}")
        return final
    last_err = None
    for attempt in range(3):
        try:
            resume = part.stat().st_size if part.exists() else 0
            if resume and expected and resume == expected:
                part.rename(final)
                print(f"\u2713 downloaded: {filename}")
                return final
            if resume and expected and resume > expected:
                part.unlink(missing_ok=True)
                resume = 0
            r = requests.get(url, headers={"Range": f"bytes={resume}-"} if resume else {},
                             stream=True, timeout=(15, 60), allow_redirects=True)
            try:
                if resume and r.status_code == 416:
                    # Range start past EOF: drop the stale .part, restart clean.
                    part.unlink(missing_ok=True)
                    r.close()
                    continue
                if resume and r.status_code == 206:
                    total = resume + int(r.headers.get("Content-Length") or 0)
                else:
                    r.raise_for_status()
                    if resume:
                        print(f"{filename}: server ignored Range, restarting")
                        part.unlink(missing_ok=True)
                        resume = 0
                    total = expected or int(r.headers.get("Content-Length") or 0)
                _stream_to(r, part, resume, total, filename)
            finally:
                r.close()
            part.rename(final)
            print(f"\u2713 downloaded: {filename}")
            return final
        except Exception as e:
            last_err = e
            print(f"retry {attempt + 1}/3 failed: {e}")
            time.sleep(5)
    raise RuntimeError(f"download failed: {url} ({last_err})")


# ---- speed fix cell 2 installs as a local Diffucore extension -----------------
# Diffucore re-stages the prompt's LoRAs for every single image: it re-reads the
# LoRA file and rebuilds every weight delta on the CPU. An unchanged <lora:…>
# set is already fused, so the extension below skips that work and reports the
# seconds it saved in the image's info line.
LORA_KEEP_MANIFEST = {
    "name": "anima-lora-keep",
    "title": "Keep prompt LoRAs applied",
    "version": "1.0.0",
    "description": "Skips Diffucore's per-image LoRA re-stage when the prompt's "
                   "<lora:...> set is unchanged.",
    "entry": "extension.py",
}

LORA_KEEP_EXT_SOURCE = r'''"""Keep prompt LoRAs applied between images (written by the anima-forge notebook).

Diffucore's engine clears and re-fuses every <lora:...> tag on every image: it
re-reads the LoRA file and rebuilds all of the weight deltas on the CPU. When the
tag set is unchanged the fused weights are already correct, so this patch skips
the rebuild and reports the time it saved.
"""

import time

_LOG = "[anima-lora-keep]"


def setup(api):
    engine = getattr(api, "engine", None)
    original = getattr(engine, "apply_temp_loras", None) if engine is not None else None
    if not callable(original):
        print(_LOG + " engine.apply_temp_loras not found - nothing patched", flush=True)
        return

    state = {"key": None, "seconds": 0.0}

    def apply_temp_loras(loras):
        key = [(str(name), float(mult)) for name, mult in loras]
        bundle = getattr(engine, "_loaded", None)
        applied = list(getattr(bundle, "applied_loras", None) or [])
        # Skip only when this exact set is already fused into this same model.
        if key and key == state["key"] and [n for n, _ in key] == applied:
            print(_LOG + f" skipped re-staging {key} (would have taken"
                  f" {state['seconds']:.1f}s)", flush=True)
            return f"LoRAs unchanged - skipped {state['seconds']:.1f}s re-stage"
        t0 = time.perf_counter()
        report = original(loras)
        state["key"] = key or None
        state["seconds"] = time.perf_counter() - t0
        print(_LOG + f" staged {key} in {state['seconds']:.1f}s", flush=True)
        return f"{report} ({state['seconds']:.1f}s)"

    engine.apply_temp_loras = apply_temp_loras
    print(_LOG + " active - unchanged prompt LoRAs stay applied between images",
          flush=True)
'''


print("\u2713 Set up done — Drive:", GDRIVE)
print("  Now run cell 2 (🚀 Start everything).")


In [ ]:
# @title 2️⃣ 🚀 Start everything — one run, safe to re-run {"form-width":"30%"}
# Order is handled internally: install → models → UI → public link → Anima load
# (T4 speed stack) → warm-up → "you're live". Re-running skips finished steps.

import json, queue, socket, subprocess, sys, threading, time
import requests

if "GDRIVE" not in globals():
    raise SystemExit("⚠ Run cell 1 first — it sets up Drive, the paths and the downloader.")

BASE = f"http://127.0.0.1:{PORT}"
HDRS = {"Authorization": f"Bearer {UI_PASSWORD.strip()}"} if UI_PASSWORD.strip() else {}
PRELOAD_TIMEOUT = 600   # seconds per load attempt (first load builds kernels + reads ~5.5 GB)


def phase(n, label):
    print(f"\n[{n}/5] {label}", flush=True)


def port_open(p, host="127.0.0.1"):
    with socket.socket() as s:
        return s.connect_ex((host, p)) == 0


# ── 1/5 install ───────────────────────────────────────────────────────────────
phase(1, "Install Diffucore UI + engine")
if not INSTALL_DEPS:
    print("      skipped (INSTALL_DEPS = False)")
else:
    if not (UI_DIR / "backend" / "app.py").exists():
        print("      first clone of the UI + engine …")
        !rm -rf $UI_DIR
        !git clone -q --recurse-submodules https://github.com/nawka12/diffucore-ui.git $UI_DIR
    else:
        !git -C $UI_DIR pull -q --ff-only || true
    !git -C $UI_DIR submodule update --init --recursive || true

    # Colab already ships a CUDA torch build — repair it only if it is missing/broken.
    probe = subprocess.run(
        [sys.executable, "-c",
         "import torch; print('CUDA_OK' if torch.cuda.is_available() else 'CUDA_BAD')"],
        capture_output=True, text=True)
    if "CUDA_OK" not in probe.stdout:
        if "ModuleNotFoundError" in (probe.stdout + probe.stderr):
            !pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu124
        else:
            print("      ⚠ torch is installed but CUDA is off — switch the runtime to a GPU")

    print("      installing dependencies (~3–5 min) …")
    !pip install -q -r $UI_DIR/requirements.txt
    !pip install -q -e $UI_DIR/diffucore
    !pip install -q pyngrok ninja  # tunnel, and the toolchain that builds the T4 int8 kernel
    !apt-get -qq update -y >/dev/null 2>&1 || true
    !apt-get -qq install -y aria2 >/dev/null 2>&1 || true   # 16-connection downloads (optional)

    # models/ and outputs/ become symlinks into Drive: downloads and every
    # generated PNG land there automatically.
    for _link, _target in ((UI_DIR / "models", DRIVE_MODELS), (UI_DIR / "outputs", DRIVE_OUTPUTS)):
        if _link.is_symlink():
            _link.unlink()
        elif _link.exists():
            shutil.rmtree(_link, ignore_errors=True)
        _link.symlink_to(_target)
    print("      done")

# ── 2/5 model files ───────────────────────────────────────────────────────────
phase(2, "Model files in Drive (files already there are skipped)")
if not FETCH_MODELS:
    print("      skipped (FETCH_MODELS = False)")
else:
    JOBS = [
        (f"{HF_ANIMA}/split_files/diffusion_models/anima-base-v1.0.safetensors",
         DRIVE_MODELS / "diffusion-models", "anima-base-v1.0.safetensors"),
        (f"{HF_ANIMA}/split_files/text_encoders/qwen_3_06b_base.safetensors",
         DRIVE_MODELS / "text-encoders", "qwen_3_06b_base.safetensors"),
        (f"{HF_ANIMA}/split_files/vae/qwen_image_vae.safetensors",
         DRIVE_MODELS / "vae", "qwen_image_vae.safetensors"),
    ]
    if TURBO_LORA:
        # Official turbo LoRA for the base DiT — enables fast 8–12 step, CFG 1 generation.
        JOBS.append((f"{HF_ANIMA_LORAS}/anima-turbo-lora-v0.2.safetensors",
                     DRIVE_MODELS / "loras", "anima-turbo-lora-v0.2.safetensors"))
    if FACE_DETAILER:
        # YOLO face detector for Diffucore UI's built-in ADetailer-style face fix.
        JOBS.append((f"{HF_ADETAILER}/face_yolov8n.pt",
                     DRIVE_MODELS / "detailers", "face_yolov8n.pt"))
    for _url, _dest, _name in JOBS:
        fetch(_url, _dest, _name)

# ── 3/5 the UI server ─────────────────────────────────────────────────────────
phase(3, "Boot the Diffucore UI")
LOG = UI_DIR / "server.log"
_reusing = port_open(PORT)
if KEEP_LORAS_APPLIED:
    _ext = UI_DIR / "extensions" / LORA_KEEP_MANIFEST["name"]
    _ext.mkdir(parents=True, exist_ok=True)
    (_ext / "extension.json").write_text(json.dumps(LORA_KEEP_MANIFEST, indent=2) + "\n")
    (_ext / "extension.py").write_text(LORA_KEEP_EXT_SOURCE)
    print("      lora-keep extension installed (unchanged prompt LoRAs skip the re-stage)"
          + (" — stop the running server from cell 4 and re-run this cell to load it"
             if _reusing else ""))
if _reusing:
    print(f"      already answering on port {PORT} — reusing it")
elif not (UI_DIR / "backend" / "app.py").exists():
    raise SystemExit(f"⚠ {UI_DIR}/backend/app.py is missing — set INSTALL_DEPS = True in cell 1,"
                     " re-run cell 1, then run this cell again.")
else:
    subprocess.run("pkill -f 'backend/app.py'", shell=True)
    time.sleep(2)
    _cmd = [sys.executable, str(UI_DIR / "backend" / "app.py"), "--listen", "--port", str(PORT)]
    if UI_PASSWORD.strip():
        _cmd += ["--auth-token", UI_PASSWORD.strip()]
    proc = subprocess.Popen(_cmd, cwd=str(UI_DIR), stdout=open(LOG, "a"),
                            stderr=subprocess.STDOUT)
    _t0 = time.time()
    while time.time() - _t0 < 240:
        if proc.poll() is not None:
            print("      ✗ the server exited — last log lines:")
            print("\n".join(LOG.read_text(errors="ignore").splitlines()[-25:]))
            raise SystemExit(1)
        if port_open(PORT):
            break
        time.sleep(2)
    else:
        print("      ✗ timed out waiting for the server — last log lines:")
        print("\n".join(LOG.read_text(errors="ignore").splitlines()[-25:]))
        raise SystemExit(1)
    print(f"      up on port {PORT} (PID {proc.pid})")

# ── 4/5 public link ───────────────────────────────────────────────────────────
phase(4, "Public link")
_token = NGROK_TOKEN.strip()
if not _token:
    try:
        from google.colab import userdata
        _token = userdata.get("NGROK_TOKEN")
    except Exception:
        _token = ""
url = None
if not _token:
    print("      ⚠ no ngrok token — add one in cell 1 (or a Colab Secret named NGROK_TOKEN)")
    print("        and re-run this cell: the server and the loaded model stay up.")
else:
    from pyngrok import ngrok
    ngrok.set_auth_token(_token)
    ngrok.kill()
    _kwargs = {"domain": NGROK_DOMAIN.strip()} if NGROK_DOMAIN.strip() else {}
    url = ngrok.connect(PORT, "http", **_kwargs).public_url
    print("      link ready:", url)

# ── 5/5 Anima load + warm-up ──────────────────────────────────────────────────
phase(5, "Load Anima with the T4 speed stack")


def _api(path, **kw):
    return requests.get(BASE + path, headers=HDRS, timeout=15, **kw)


# The startup hash scan can delay the first API response.
deadline = time.time() + 180
while True:
    try:
        _api("/api/status"); break
    except Exception:
        if time.time() > deadline:
            raise SystemExit("Server API did not come up — check cell 2's server output.")
        time.sleep(2)

models = _api("/api/models").json()

# Reuse the last successful load form (files chosen in a previous run) when present.
lf = models.get("load_form") or {}
PRELOAD_DIT = lf.get("dit") or "anima-base-v1.0.safetensors"
PRELOAD_VAE = lf.get("vae") or "qwen_image_vae.safetensors"
PRELOAD_TE = lf.get("te") or "qwen_3_06b_base.safetensors"

missing = [n for n, key in ((PRELOAD_DIT, "dits"), (PRELOAD_VAE, "vaes"), (PRELOAD_TE, "tes"))
           if n not in models.get(key, [])]
if missing:
    raise SystemExit(f"Not on disk yet: {missing} — re-run cell 2 with FETCH_MODELS = True.")

# ---- form defaults: fast params + default negative prompt (seeded into the Generate form) ----
if FAST_PARAMS or DEFAULT_NEG.strip():
    st = _api("/api/settings").json()
    gd = st.get("gen_defaults") or {}
    if FAST_PARAMS:
        st["cfg_interval_start"] = 0.1   # skip the negative pass outside 0.1–0.9
        st["cfg_interval_end"] = 0.9     # -> ~10% fewer forward passes (guide: opt-in speedup)
        gd.update({
            "sampler": "dpmpp_2m",       # guide: stays clean at 16–20 steps where
            "scheduler": "beta",         # er_sde wants 24–30 for the same quality
            "steps": 20, "cfg": 4.0, "width": 832, "height": 1216, "shift": 3.0,
        })
    if DEFAULT_NEG.strip():
        # Pin the negative prompt; null/absent leaves the form's own value alone.
        gd["neg"] = DEFAULT_NEG.strip()
    st["gen_defaults"] = gd
    r = requests.post(BASE + "/api/settings", json=st, headers=HDRS, timeout=30)
    if r.status_code == 200:
        if FAST_PARAMS:
            print("      ⚙ fast params armed: dpmpp_2m/beta · 20 steps · CFG 4.0 · 832×1216 · CFG interval 0.1–0.9")
        if DEFAULT_NEG.strip():
            print(f"      🚫 default negative prompt armed ({len(DEFAULT_NEG.strip())} chars)")
    else:
        print(f"      ⚠ could not save form defaults ({r.status_code}) — continuing with UI defaults")

if not PRELOAD:
    print("      skipped (PRELOAD = False) — load the model from the UI's top bar instead.")
else:
    # ---- capability-gated attempt ladder ----
    int8_ok = bool(models.get("int8_available"))
    fuse_ok = bool(models.get("fused_glue_available"))
    if USE_TORCH_COMPILE:
        plan = [(False, False, True, "torch.compile + fp16 accumulate + fp16 VAE")]
    elif FAST_TURING_KERNELS and int8_ok:
        plan = [("int8_turing", fuse_ok, False,
                 "int8 attention + fp16 accumulate + fp16 VAE" + (" + fused glue" if fuse_ok else "")),
                ("sdpa", fuse_ok, False,
                 "fp16 accumulate + fp16 VAE" + (" + fused glue" if fuse_ok else " (int8 kernel unavailable)")),
                ("sdpa", False, False, "stock SDPA (all Turing kernels off)")]
    elif FAST_TURING_KERNELS and fuse_ok:
        plan = [("sdpa", True, False, "fused glue + fp16 accumulate + fp16 VAE"),
                ("sdpa", False, False, "stock SDPA")]
    else:
        plan = [("sdpa", False, False, "fp16 accumulate + fp16 VAE (Turing kernels unavailable here)")]

    # ---- SSE listener: captures the load job's done/error event ----
    events = queue.Queue()

    def _listen():
        try:
            with requests.get(BASE + "/api/events", headers=HDRS, stream=True,
                              timeout=(10, 3600)) as r:
                for line in r.iter_lines(decode_unicode=True):
                    if line and line.startswith("data:"):
                        try:
                            events.put(json.loads(line[5:].strip()))
                        except ValueError:
                            pass
        except Exception:
            pass

    threading.Thread(target=_listen, daemon=True).start()
    time.sleep(1.5)  # let the stream connect before the first POST

    def _wait_job(job_id, timeout, expect_load=True):
        """Wait for this job's SSE event (and, for loads, the status flip).
        Returns (ok, event_or_detail)."""
        t0 = time.time()
        while time.time() - t0 < timeout:
            if expect_load:
                try:
                    st_ = _api("/api/status").json().get("status", "")
                except Exception:
                    st_ = ""
                if "Anima(" in st_:
                    return True, st_
            try:
                ev = events.get(timeout=2)
            except queue.Empty:
                continue
            if ev.get("type") == "error" and ev.get("job") == job_id:
                return False, f"load failed: {ev.get('message', '?')}"
            if ev.get("type") == "done" and ev.get("job") == job_id:
                if not expect_load:
                    return True, ev
                time.sleep(1)
                try:
                    st_ = _api("/api/status").json().get("status", "")
                except Exception:
                    st_ = ""
                if "Anima(" in st_:
                    return True, st_
        return False, "timed out"

    payload_base = {"model_type": "Anima", "dit": PRELOAD_DIT, "vae": PRELOAD_VAE,
                    "te": PRELOAD_TE,
                    "offload": "none" if RESIDENT_ENCODERS else None,
                    "cuda_graphs": False, "channels_last": False, "tf32": False,
                    "fp16_accumulation": True, "vae_fp16": True}
    ok = False
    for attention, fused_glue, compile_, label in plan:
        payload = {**payload_base, "attention": attention,
                   "fused_glue": fused_glue, "compile": compile_}
        r = requests.post(BASE + "/api/load", json=payload, headers=HDRS, timeout=30)
        if r.status_code != 200:
            print(f"      load rejected ({r.status_code}) — trying safer flags: {r.text[:200]}")
            continue
        job_id = r.json().get("job")
        print(f"      ⏳ loading {PRELOAD_DIT} — {label} (up to {PRELOAD_TIMEOUT}s) …")
        ok, detail = _wait_job(job_id, PRELOAD_TIMEOUT)
        if ok:
            print("      ⚡", detail)
            print(f"      speed stack: {label}" + (" · encoders resident" if RESIDENT_ENCODERS else ""))
            break
        print(f"      ↳ {detail}; falling back to safer flags")

    # ---- warmup: absorb kernel JIT / cuDNN autotune before the user's first click ----
    if ok and WARMUP:
        warm = {"mode": "t2i", "prompt": "a plain gray cube on a white table, soft studio light",
                "neg": "", "sampler": "dpmpp_2m", "scheduler": "beta", "steps": 2, "cfg": 4.0,
                "seed": 0, "width": 832, "height": 1216, "shift": 3.0,
                "preview": False, "blur_check": False}
        try:
            r = requests.post(BASE + "/api/generate", json=warm, headers=HDRS, timeout=30)
            if r.status_code == 200:
                _t0 = time.time()
                wok, ev = _wait_job(r.json().get("job"), PRELOAD_TIMEOUT, expect_load=False)
                if wok and isinstance(ev, dict) and ev.get("path"):
                    print(f"      🔥 warmup done in {time.time() - _t0:.1f}s (throwaway image removed)")
                    try:
                        requests.delete(BASE + "/api/gallery",
                                        params={"path": ev["path"]}, headers=HDRS, timeout=15)
                    except Exception:
                        pass
                else:
                    print(f"      ⚠ warmup skipped ({ev}) — the first image warms up instead")
            else:
                print(f"      ⚠ warmup skipped ({r.status_code}) — the first image warms up instead")
        except Exception as e:
            print(f"      ⚠ warmup skipped ({e}) — the first image warms up instead")

    if not ok:
        print("      ✗ the model did not load. Last server log lines:")
        print("\n".join((UI_DIR / "server.log").read_text(errors="ignore").splitlines()[-25:]))
        print("      Load it manually from the UI's top bar — the speed flags sit next to Load.")

# ── you're live ───────────────────────────────────────────────────────────────
print()
print("═" * 62)
print("  🎞️  Anima / Diffucore UI :", url or f"local only (no tunnel) — port {PORT}")
if UI_PASSWORD.strip():
    print("  🔑  Password            :", UI_PASSWORD)
print("  🖼️  Images auto-save to :", DRIVE_OUTPUTS)
print("═" * 62)
print("  Generate in the browser — re-run this cell any time, it skips what's done.")


In [ ]:
# @title 3️⃣ Add a LoRA / checkpoint (optional — works mid-session) {"form-width":"30%"}
# Paste one URL per line. Civitai links should look like either:
#   https://civitai.com/api/download/models/<versionId>?token=<optional>
#   https://civitai.com/models/<id>?modelVersionId=<versionId>
# Gated models need a token: civitai.com → Account settings → API keys.

CIVITAI_TOKEN = ""  # @param {type:"string"}
FILE_URLS = ""      # @param {type:"string"}
DESTINATION = "loras"  # @param ["loras", "diffusion-models", "checkpoints", "vae", "text-encoders", "upscalers", "detailers"]
CUSTOM_FILENAME = ""  # @param {type:"string"}  (optional; only used for a single URL)

import re

assert "fetch" in globals(), "Run cell 1 first — it sets up Drive, the paths and the downloader."

lines = [l.strip() for l in FILE_URLS.splitlines() if l.strip() and not l.strip().startswith("#")]

if not lines:
    print("Nothing to import — paste a Civitai / HuggingFace URL above and re-run this cell.")
else:
    if len(lines) > 1 and CUSTOM_FILENAME.strip():
        print("(CUSTOM_FILENAME applies to a single URL — ignoring it for multiple URLs.)")
        CUSTOM_FILENAME = ""
    for url in lines:
        name = CUSTOM_FILENAME.strip() or None
        if "civitai.com" in url:
            if "/api/download/" not in url:
                m = re.search(r"modelVersionId=(\d+)", url)
                if not m:
                    raise SystemExit(
                        f"Unsupported Civitai URL: {url}\nUse the download button's link "
                        "(civitai.com/api/download/models/... or ...?modelVersionId=...).")
                url = f"https://civitai.com/api/download/models/{m.group(1)}"
            token = CIVITAI_TOKEN.strip()
            if token and "token=" not in url:
                url += ("&" if "?" in url else "?") + "token=" + token
        fetch(url, DRIVE_MODELS / DESTINATION, name)
    print(f"\n✓ saved to {DRIVE_MODELS / DESTINATION}")
    print("Refresh the UI (F5) — new files appear in the dropdowns, no restart needed.")


In [ ]:
# @title 4️⃣ Stop / status {"form-width":"30%"}
import subprocess
from pathlib import Path

ACTION = "status"  # @param ["status", "stop server"]
_UI_DIR = globals().get("UI_DIR", Path("/content/diffucore-ui"))

_pids = subprocess.run(["pgrep", "-f", "backend/app.py"],
                       capture_output=True, text=True).stdout.split()

if ACTION == "stop server":
    subprocess.run("pkill -f 'backend/app.py'", shell=True)
    print("Server stopped — run cell 2 to relaunch.")
else:
    print("Server:", f"running (PID {', '.join(_pids)})" if _pids else "stopped")
    _log = _UI_DIR / "server.log"
    if _pids and _log.exists():
        print("\n--- last log lines ---")
        print("\n".join(_log.read_text(errors="ignore").splitlines()[-15:]))
    try:
        from pyngrok import ngrok
        for t in ngrok.get_tunnels():
            print("Tunnel:", t.public_url)
    except Exception:
        pass


---
## 📋 Cheat sheet

**Generating:** open the link → describe the scene in a full sentence (Anima is
LLM-conditioned, so tag soup gives weak images) → **Generate**. Cell 2 already picked
the model, the fast sampler settings and the negative prompt for you.

| | Base model | Turbo (LoRA at ~0.8) |
|---|---|---|
| Sampler / scheduler | `dpmpp_2m` / `beta` @ 20 steps | `euler` / `flow` |
| Steps | 20 (28–32 with `er_sde` for max quality) | 8–12 |
| CFG | 4.0 | 1.0 |
| Size | 832×1216 or 1024×1024 | same |
| Negative prompt | the `DEFAULT_NEG` list from cell 1 | same (CFG 1.0 ignores it) |

- **Negative prompt:** edit `DEFAULT_NEG` in cell 1 and re-run cell 2; clear it to leave the box alone.
- **LoRAs in the prompt:** `<lora:my-lora:0.8>` — no extra step.
- **img2img** strength ~0.2–0.4 · **inpaint** denoise ~0.9–1.0 · **face fix** = the Detailer toggle.
- **Speed already on:** int8 attention + fused glue, fp16 accumulate + fp16 VAE, resident encoders,
  CFG interval 0.1–0.9, and the warm-up image cell 2 generates.
- **Prompt LoRAs stop re-staging:** cell 2 writes a tiny local extension that keeps an
  unchanged `<lora:…>` set fused between images and reports the saving in the image info
  (`LoRAs unchanged - skipped 6.4s re-stage`). Diffucore otherwise re-reads the LoRA file
  and rebuilds every weight delta on the CPU before *each* image — that is the work that
  made the panel sit on “Starting…” for several seconds. Set `KEEP_LORAS_APPLIED = False`
  in cell 1 to disable it.
- **Add a model mid-session:** cell 3 → paste the URL → F5 in the UI. No restart.
- **Where things live:** `MyDrive/AnimaDiffucore/models/…` and `outputs/YYYY-MM-DD/*.png` —
  both in Drive, both survive the runtime resetting.
- **Slow between images?** Turn *Blur NSFW* off in the page — otherwise a ~0.5 s rating job
  can sit between generations (images then aren't auto-rated).
- **OOM above ~1536²?** Set `RESIDENT_ENCODERS = False` in cell 1 and re-run cell 2.
- **Runtime idled out?** Re-run cell 1 then cell 2 — models and images are safe in Drive.
